# **LARYN FOOD WEBSITE**


Merupakan Sistem manajemen inventory dan keuangan terbaik untuk bisnis makanan Anda. Kelola stock, transaksi, dan laporan dengan mudah.

**Link URL Website:**

https://larynfood.com/


Screenshot 2026-05-10 at 02.10.19.png



*   **Functional Testing**


In [ ]:
# 1. Update sistem dan install paket dependensi dasar
!apt-get update
!apt-get install -y wget curl unzip libu2f-udev

# 2. Tambahkan kunci repositori Debian untuk mendapatkan Chromium yang kompatibel
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list

# 3. Hapus chromium-browser bawaan yang menyebabkan error "exited abnormally"
!apt-get remove -y chromium-browser

# 4. Install Google Chrome Stable (bukan Chromium)
!apt-get update
!apt-get install -y google-chrome-stable

# 5. Pasang webdriver-manager untuk otomatisasi penyamaan versi
!pip install webdriver-manager selenium pandas pytest

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [91.2 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,294 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Package

In [ ]:
import pytest
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.action_chains import ActionChains
import re
import requests

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
def setup_colab_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless') # WAJIB di Colab
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

# Contoh penggunaan sederhana di satu sel Colab:
driver = setup_colab_driver()
wait = WebDriverWait(driver, 30)

# **FITUR AUTENTIKASI**

In [ ]:

def test_login_positive(driver):
    driver.get("https://larynfood.com/login")
    email = input("Masukkan Email Admin: ")
    password = input("Masukkan Password Anda: ")

    email_input = "input[name='email']"
    password_input = "input[name='password']"
    driver.find_element(By.CSS_SELECTOR, email_input).send_keys(email)
    driver.find_element(By.CSS_SELECTOR, password_input).send_keys(password)

    submit_button = "button[type='submit']"
    driver.find_element(By.CSS_SELECTOR, submit_button).click()


    # Pastikan URL berubah ke dashboard
    success = wait.until(EC.url_contains("/dashboard"))

    assert success
    print("✅✅✅ Skenario Login Berhasil dengan waktu " + ' ' + str(time.time()) + ' Detik')

test_login_positive(driver)

time.sleep(5)


Masukkan Email Admin: Ryan@gmail.com
Masukkan Password Anda: Kepong333!
✅✅✅ Skenario Login Berhasil dengan waktu  1778380436.8286548 Detik


In [31]:

def test_login_isNotData(driver):
    driver.get("https://larynfood.com/login")
    email = input("Masukkan Email Admin: ")
    password = input("Masukkan Password Anda: ")

    submit_button = "button[type='submit']"
    driver.find_element(By.CSS_SELECTOR, submit_button).click()
    try:

        wait = WebDriverWait(driver, 5)

        # 1. Pastikan URL TIDAK berubah ke dashboard
        is_still_on_login = wait.until(EC.url_contains("/login"))

        # 2. Opsional: Cek apakah muncul pesan error di UI
        # Biasanya Laravel/Inertia akan memunculkan pesan validasi
        # error_message = driver.find_element(By.CSS_SELECTOR, ".text-red-500").text

        assert is_still_on_login
        print("✅ Skenario Input Kosong Berhasil: Sistem menolak login tanpa data.")

    except Exception as e:

        print(
            "❌ Gagal kirim request"
        )


        print(e)

        return

test_login_isNotData(driver)

time.sleep(5)


Masukkan Email Admin: 
Masukkan Password Anda: 
✅ Skenario Input Kosong Berhasil: Sistem menolak login tanpa data.


In [29]:

def test_login_isDataInvalid(driver):
    driver.get("https://larynfood.com/login")
    email = input("Masukkan Email Admin: ")
    password = input("Masukkan Password Anda: ")

    email_input = "input[name='email']"
    password_input = "input[name='password']"
    driver.find_element(By.CSS_SELECTOR, email_input).send_keys(email)
    driver.find_element(By.CSS_SELECTOR, password_input).send_keys(password)

    submit_button = "button[type='submit']"
    driver.find_element(By.CSS_SELECTOR, submit_button).click()
    try:

        wait = WebDriverWait(driver, 5)

            # Pastikan URL berubah ke dashboard
        success = wait.until(EC.url_contains("/dashboard"))
        print("✅ Skenario Input Kosong Berhasil: Sistem menolak login tanpa data.")

    except Exception as e:

        print(
            "❌ Gagal kirim request"
        )

        alert_element = driver.find_element(By.CSS_SELECTOR, ".alert-danger")
        print(f"⚠️ Pesan Alert: {alert_element.text}")
        print("✅ Skenario Input Salah Data Tidak Ditemukan Berhasil: Sistem menolak login tanpa data.")

        return

test_login_isDataInvalid(driver)

time.sleep(5)


Masukkan Email Admin: muhammad@gmail.com
Masukkan Password Anda: 12345678
❌ Gagal kirim request
⚠️ Pesan Alert: Email atau password salah.
✅ Skenario Input Salah Data Tidak Ditemukan Berhasil: Sistem menolak login tanpa data.


# **FITUR KATEGORI**

**Daftar Kategori**

In [ ]:
import pandas as pd
import re # Import Regex untuk ambil angka dari URL

time.sleep(5)

def test_list_categories_with_id(driver):
    driver.get("https://larynfood.com/categories")
    all_data = []

    while True:
        soup = BeautifulSoup(driver.page_source, 'lxml')
        table = soup.find('table', class_='table')

        if not table: break

        rows = table.find_all('tr')[1:] # Lewati header

        for row in rows:
            cols = row.find_all('td')
            if cols:

                nama = cols[1].get_text(strip=True)
                tipe = cols[2].get_text(strip=True)


                link_edit = cols[4].find('a', href=re.compile(r'/categories/\d+/edit'))

                category_id = None
                if link_edit:
                    href = link_edit.get('href')
                    # Mengambil angka saja dari URL menggunakan regex
                    # Contoh: '.../categories/12/edit' -> '12'
                    match = re.search(r'/categories/(\d+)/edit', href)
                    if match:
                        category_id = match.group(1)

                all_data.append([category_id, nama, tipe])

        for _ in range(10):
            driver.execute_script("window.scrollBy(0, 500);")
            time.sleep(0.3)

        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")

            driver.execute_script("arguments[0].scrollIntoView();", next_button)
            next_button.click()
        except:

            print("Selesai mengambil semua halaman.")
            break


    df = pd.DataFrame(all_data, columns=["ID", "Nama Kategori", "Tipe"])
    return df

test_list_categories_with_id(driver)

Selesai mengambil semua halaman.


,ID,Nama Kategori,Tipe
0,29,Biji Wijen,Bahan Baku
1,5,Chili oil,Bahan Baku
2,10,Dimsum Goreng,Bahan Baku
3,3,Dimsum kukus,Bahan Baku
4,9,Garnish (hiasan makanan),Bahan Baku
5,8,Keju,Bahan Baku
6,14,Kemasan Box,Peralatan / Kemasan
7,16,Kemasan kresek,Peralatan / Kemasan
8,13,Kemasan Tray Aluminium,Peralatan / Kemasan
9,17,Kresek plastik,Peralatan / Kemasan


**Tambah Kategori**

In [ ]:
def test_add_categories_item(driver):

    driver.get("https://larynfood.com/categories")

    clickTombolKategori = "a[href^='https://larynfood.com/categories/create']"
    driver.find_element(By.CSS_SELECTOR, clickTombolKategori).click()
    print("✅ Klik tombol kategori berhasil")

    kategori = input("Masukkan Kategori Produk: ")
    kategori_input = "input[name='nama_kategori']"

    pilih_radio = input("Pilih Kategori (produk atau peralatan): ")
    xpath = f"//input[@type='radio' and @value='{pilih_radio}']"
    try:
        radio = driver.find_element(By.XPATH, xpath)
        if not radio.is_selected():
            radio.click()
            print(f"Berhasil memilih radio dengan value: {pilih_radio}")
    except:
        print(f"Opsi {pilih_radio} tidak ditemukan!")

    deskripsi = input("Masukkan Deskripsi Produk: ")
    deskripsi_input = "textarea[name='deskripsi']"

    driver.find_element(By.CSS_SELECTOR, kategori_input).send_keys(kategori)


    driver.find_element(By.CSS_SELECTOR, deskripsi_input).send_keys(deskripsi)

    session = requests.Session()


    # =========================
    # AMBIL CSRF TOKEN
    # =========================
    try:

        csrf_token = (
            driver.find_element(
                By.NAME,
                "_token"
            ).get_attribute("value")
        )

        print("✅ CSRF TOKEN:")
        print(csrf_token)

    except Exception as e:

        print(
            "❌ Gagal ambil CSRF token"
        )

        print(e)

        return

    # =========================
    # COPY COOKIES SELENIUM
    # =========================
    selenium_cookies = (
        driver.get_cookies()
    )

    for cookie in selenium_cookies:

        session.cookies.set(
            cookie["name"],
            cookie["value"]
        )

    print(
        "✅ Cookies berhasil dipindahkan"
    )

    # =========================
    # UPDATE URL
    # =========================
    update_url = (
        f"https://larynfood.com/"
        f"categories"
    )

    print("UPDATE URL:")
    print(update_url)

    payload = {

        "_token": csrf_token,

        "_method": "POST",

        "nama_kategori": kategori,

        "jenis_kategori":
            pilih_radio,

        "deskripsi":
            deskripsi
    }

    # =========================
    # HEADERS
    # =========================
    headers = {

        "Referer":
            driver.current_url,

        "User-Agent":
            driver.execute_script(
                "return navigator.userAgent;"
            ),

        "X-Requested-With":
            "XMLHttpRequest"
    }

    # =========================
    # SEND REQUEST
    # =========================
    try:

        response = session.post(
            update_url,
            data=payload,
            headers=headers,
            allow_redirects=True
        )

        print(
            "✅ Request update terkirim"
        )

    except Exception as e:

        print(
            "❌ Gagal kirim request"
        )

        print(e)

        return

    # =========================
    # DEBUG RESPONSE
    # =========================
    print("========== RESPONSE ==========")

    print(
        "STATUS CODE:",
        response.status_code
    )

    print(
        "FINAL URL:",
        response.url
    )

    print("==============================")

    # =========================
    # VALIDASI RESULT
    # =========================
    if "/login" in response.url:

        print(
            "❌ Redirect ke LOGIN"
        )

        print(
            "⚠️ Session/auth middleware "
            "masih ditolak"
        )

        with open(
            "response_login.html",
            "w",
            encoding="utf-8"
        ) as f:

            f.write(response.text)

        print(
            "✅ HTML response disimpan "
            "ke response_login.html"
        )

        return

    elif response.status_code == 200:

        print("✅✅✅ Skenario Tambah Kategori Berhasil dengan waktu " + ' ' + str(time.time()) + ' Detik')

        return

    else:

        print(
            "⚠️ Response tidak dikenali"
        )

        print(response.text[:1000])

        return

test_add_categories_item(driver)

✅ Klik tombol kategori berhasil
Masukkan Kategori Produk: Aluminium Foil
Pilih Kategori (produk atau peralatan): peralatan
Berhasil memilih radio dengan value: peralatan
Masukkan Deskripsi Produk: -
✅ CSRF TOKEN:
XD0oYw5JKRCp1qMG18cKYzJLOJmJWOoM0pjTWWBs
✅ Cookies berhasil dipindahkan
UPDATE URL:
https://larynfood.com/categories
✅ Request update terkirim
========== RESPONSE ==========
STATUS CODE: 200
FINAL URL: https://larynfood.com/categories
✅✅✅ Skenario Tambah Kategori Berhasil dengan waktu  1778380515.2797625 Detik


**Edit Kategori**

In [ ]:
def test_edit_categories_with_id(driver):
    driver.get("https://larynfood.com/categories")

    names = input("Masukkan kategori yang ingin diedit: ")

    soup = BeautifulSoup(driver.page_source, 'lxml')
    table = soup.find('table', class_='table')



    rows = table.find_all('tr')[1:]

    for row in rows:
        cols = row.find_all('td')
        if not cols: continue

        id = cols[0].get_text(strip=True)
        nama = cols[1].get_text(strip=True)

        if nama == names:
            print(f"Match ditemukan: {nama}")
            link_edit = cols[4].find('a', href=re.compile(r'/categories/\d+/edit'))

            category_id = None
            if link_edit:
                href = link_edit.get('href')
                # Mengambil angka saja dari URL menggunakan regex
                # Contoh: '.../categories/12/edit' -> '12'
                match = re.search(r'/categories/(\d+)/edit', href)
                if match:
                    category_id = match.group(1)

            xpath_edit = f"//td[text()='{names}']/parent::tr//a[starts-with(@href, 'https://larynfood.com/categories/{category_id}/edit')]"


            edit_button = driver.find_element(By.XPATH, xpath_edit)

            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", edit_button)
            time.sleep(0.5)

            edit_button.click()

            print("✅ Berhasil masuk halaman edit kategori")



            kategori_input = "input[name='nama_kategori']"


            deskripsi_input = "textarea[name='deskripsi']"

            input_nama = WebDriverWait(driver, 10).until(
EC.element_to_be_clickable((By.NAME, "nama_kategori"))
)
            input_nama.clear()  # WAJIB: Hapus teks lama dulu
            kategori = input("Masukkan Kategori Produk: ")
            input_nama.send_keys(kategori)

            pilih_radio = input("Pilih Kategori (produk atau peralatan): ")


            xpath_radio = f"//input[@name='jenis_kategori' and @value='{pilih_radio}']"
            try:
                radio = driver.find_element(By.XPATH, xpath_radio)
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", radio)
                radio.click()
            except:
                print(f"⚠️ Opsi radio {pilih_radio} tidak ditemukan, cek huruf besar/kecil value-nya!")

            # 3. Clear dan Isi Deskripsi
            input_desc = driver.find_element(By.NAME, "deskripsi")
            input_desc.clear()  # WAJIB: Hapus teks lama dulu
            deskripsi = input("Masukkan Deskripsi Produk: ")
            input_desc.send_keys(deskripsi)

            session = requests.Session()


            # =========================
            # AMBIL CSRF TOKEN
            # =========================
            try:

                csrf_token = (
                    driver.find_element(
                        By.NAME,
                        "_token"
                    ).get_attribute("value")
                )

                print("✅ CSRF TOKEN:")
                print(csrf_token)

            except Exception as e:

                print(
                    "❌ Gagal ambil CSRF token"
                )

                print(e)

                return

            # =========================
            # COPY COOKIES SELENIUM
            # =========================
            selenium_cookies = (
                driver.get_cookies()
            )

            for cookie in selenium_cookies:

                session.cookies.set(
                    cookie["name"],
                    cookie["value"]
                )

            print(
                "✅ Cookies berhasil dipindahkan"
            )

            # =========================
            # UPDATE URL
            # =========================
            update_url = (
                f"https://larynfood.com/"
                f"categories/{category_id}"
            )

            print("UPDATE URL:")
            print(update_url)

            payload = {

                "_token": csrf_token,

                "_method": "PUT",

                "nama_kategori": kategori,

                "jenis_kategori":
                    pilih_radio,

                "deskripsi":
                    deskripsi
            }

            # =========================
            # HEADERS
            # =========================
            headers = {

                "Referer":
                    driver.current_url,

                "User-Agent":
                    driver.execute_script(
                        "return navigator.userAgent;"
                    ),

                "X-Requested-With":
                    "XMLHttpRequest"
            }

            # =========================
            # SEND REQUEST
            # =========================
            try:

                response = session.post(
                    update_url,
                    data=payload,
                    headers=headers,
                    allow_redirects=True
                )

                print(
                    "✅ Request update terkirim"
                )

            except Exception as e:

                print(
                    "❌ Gagal kirim request"
                )

                print(e)

                return

            # =========================
            # DEBUG RESPONSE
            # =========================
            print("========== RESPONSE ==========")

            print(
                "STATUS CODE:",
                response.status_code
            )

            print(
                "FINAL URL:",
                response.url
            )

            print("==============================")

            # =========================
            # VALIDASI RESULT
            # =========================
            if "/login" in response.url:

                print(
                    "❌ Redirect ke LOGIN"
                )

                print(
                    "⚠️ Session/auth middleware "
                    "masih ditolak"
                )

                with open(
                    "response_login.html",
                    "w",
                    encoding="utf-8"
                ) as f:

                    f.write(response.text)

                print(
                    "✅ HTML response disimpan "
                    "ke response_login.html"
                )

                return

            elif response.status_code == 200:

                print("✅✅✅ Skenario Edit Kategori Berhasil dengan waktu " + ' ' + str(time.time()) + ' Detik')


                return

            else:

                print(
                    "⚠️ Response tidak dikenali"
                )

                print(response.text[:1000])

                return


test_edit_categories_with_id(driver)

Masukkan kategori yang ingin diedit: Aluminium Foil
Match ditemukan: Aluminium Foil
✅ Berhasil masuk halaman edit kategori
Masukkan Kategori Produk: Aluminium
Pilih Kategori (produk atau peralatan): peralatan
Masukkan Deskripsi Produk: -
✅ CSRF TOKEN:
XD0oYw5JKRCp1qMG18cKYzJLOJmJWOoM0pjTWWBs
✅ Cookies berhasil dipindahkan
UPDATE URL:
https://larynfood.com/categories/33
✅ Request update terkirim
========== RESPONSE ==========
STATUS CODE: 200
FINAL URL: https://larynfood.com/categories
✅✅✅ Skenario Edit Kategori Berhasil dengan waktu  1778380543.628841 Detik


**Hapus Kategori**

In [ ]:
def test_delete_categories_with_id(driver):
    driver.get("https://larynfood.com/categories")

    names = input("Masukkan kategori yang ingin dihapus: ")
    found = False

    while True:
        soup = BeautifulSoup(driver.page_source, 'lxml')
        table = soup.find('table', class_='table')

        if not table: break

        rows = table.find_all('tr')[1:]

        for row in rows:
            cols = row.find_all('td')
            if not cols: continue

            nama = cols[1].get_text(strip=True)

            if nama == names:
                print(f"Match ditemukan: {nama}")

                xpath_delete = f"//td[text()='{names}']/parent::tr//button[@type='submit']"
                delete_button = driver.find_element(By.XPATH, xpath_delete)

                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", delete_button)
                time.sleep(0.5)

                delete_button.click()

                try:
                    WebDriverWait(driver, 5).until(EC.alert_is_present())
                    alert = driver.switch_to.alert
                    print(f"Isi Alert: {alert.text}")
                    message_input = input('Ok atau Cancel ')
                    if message_input == 'Ok':
                      alert.accept() # Klik OK
                    else:
                      alert.dismiss() # Klik Cancel
                    print(f"🗑️ Kategori '{names}' berhasil dihapus.")
                    found = True
                except:
                    print("Tidak ada alert konfirmasi.")

                time.sleep(2)
                print("✅✅✅ Skenario Hapus Kategori Berhasil dengan waktu" + ' ' + str(time.time()) + ' Detik')
                return

        # Logika Pagination: Jika tidak ketemu di halaman ini, coba klik 'Next'
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")
            next_button.click()
            time.sleep(2) # Tunggu loading halaman berikutnya
        except:
            print("Nama tidak ditemukan hingga halaman terakhir.")
            break


test_delete_categories_with_id(driver)

Masukkan kategori yang ingin dihapus: Aluminium
Match ditemukan: Aluminium
Isi Alert: Apakah Anda yakin ingin menghapus kategori ini?
Ok atau Cancel Ok
🗑️ Kategori 'Aluminium' berhasil dihapus.
✅✅✅ Skenario Hapus Kategori Berhasil dengan waktu 1778380561.3709133 Detik


# **FITUR SUPPLIER**

**Daftar Supplier**

In [ ]:
import pandas as pd
import re # Import Regex untuk ambil angka dari URL

time.sleep(5)

def test_list_supplier_with_id(driver):
    driver.get("https://larynfood.com/suppliers")
    all_data = []

    while True:
        soup = BeautifulSoup(driver.page_source, 'lxml')
        table = soup.find('table', class_='table')

        if not table: break

        rows = table.find_all('tr')[1:] # Lewati header

        for row in rows:
            cols = row.find_all('td')
            if cols:

                nama_supplier = cols[0].get_text(strip=True)
                kontak = cols[1].get_text(strip=True)
                kota = cols[2].get_text(strip=True)
                email = cols[4].get_text(strip=True)


                link_edit = cols[5].find('a', href=re.compile(r'/suppliers/\d+/edit'))

                supplier_id = None
                if link_edit:
                    href = link_edit.get('href')

                    match = re.search(r'/suppliers/(\d+)/edit', href)
                    if match:
                        supplier_id = match.group(1)

                all_data.append([supplier_id, nama_supplier, kontak, kota, email])

        for _ in range(10):
            driver.execute_script("window.scrollBy(0, 500);")
            time.sleep(0.3)

        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")

            driver.execute_script("arguments[0].scrollIntoView();", next_button)
            next_button.click()
        except:

            print("Selesai mengambil semua halaman.")
            break


    df = pd.DataFrame(all_data, columns=["ID", "Nama Supplier", "Kontak", "Kota", "Email"])
    return df

test_list_supplier_with_id(driver)

Selesai mengambil semua halaman.


,ID,Nama Supplier,Kontak,Kota,Email
0,7,INCLUDED,Anna,Sidoarjo,Incip@gmail.com
1,6,Nisrina Foods,-,Sidoarjo,nisrinafood@gmail.com
2,5,Toko Nico,-,Sidoarjo,nico@gmail.com
3,4,Toko pasar sepanjang,-,Sidoarjo,sepanjang@gmail.com
4,3,Toko Shopee,-,Shopee,shopee@gmail.com
5,2,YOIKI Frozen Food,-,Surabaya,umifrozen.97@gmail.com


**Tambah Supplier**

In [ ]:
def test_add_suppliers_item(driver):

    driver.get("https://larynfood.com/suppliers")

    clickTombolSuppliers = "a[href^='https://larynfood.com/suppliers/create']"
    wait = WebDriverWait(driver, 10)

    # Tunggu sampai elemen ada di DOM
    element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, clickTombolSuppliers)))

    # Klik menggunakan JavaScript
    driver.execute_script("arguments[0].click();", element)
    print("✅ Klik tombol suppliers berhasil")

    supplier = input("Masukkan Nama Supplier: ")
    supplier_input = "input[name='nama_supplier']"

    kontak = input("Masukkan Kontak yang bisa dihubungi: ")
    kontak_input = "input[name='kontak']"

    nomor_telepon = input("Masukkan Nomor Telepon yang bisa dihubungi: ")
    telepon_input = "input[name='nomor_telepon']"

    email = input("Masukkan Email yang bisa dihubungi: ")
    email_input = "input[name='email']"

    alamat = input("Masukkan Alamat yang bisa dirujuk: ")
    alamat_input = "textarea[name='alamat']"

    kota = input("Masukkan Kota Anda: ")
    alamat_input = "input[name='kota']"

    data_supplier = {
    supplier: supplier_input,
    kontak: kontak_input,
    nomor_telepon: telepon_input,
    email: email_input,
    alamat: alamat_input,
    kota: alamat_input
    }


    # Jika ingin dimasukkan ke dalam satu Array (List)
    daftar_supplier = [data_supplier]

    for data in daftar_supplier:
        for nama, input_field in data.items():
            driver.find_element(By.CSS_SELECTOR, input_field).send_keys(nama)


    session = requests.Session()


    # =========================
    # AMBIL CSRF TOKEN
    # =========================
    try:

        csrf_token = (
            driver.find_element(
                By.NAME,
                "_token"
            ).get_attribute("value")
        )

        print("✅ CSRF TOKEN:")
        print(csrf_token)

    except Exception as e:

        print(
            "❌ Gagal ambil CSRF token"
        )

        print(e)

        return

    # =========================
    # COPY COOKIES SELENIUM
    # =========================
    selenium_cookies = (
        driver.get_cookies()
    )

    for cookie in selenium_cookies:

        session.cookies.set(
            cookie["name"],
            cookie["value"]
        )

    print(
        "✅ Cookies berhasil dipindahkan"
    )

    # =========================
    # UPDATE URL
    # =========================
    update_url = (
        f"https://larynfood.com/"
        f"suppliers"
    )

    print("UPDATE URL:")
    print(update_url)

    payload = {

        "_token": csrf_token,

        "_method": "POST",

        "nama_supplier": supplier,

        "kontak": kontak,

        "nomor_telepon": nomor_telepon,

        "email": email,

        "alamat": alamat,

        "kota": kota

    }

    # =========================
    # HEADERS
    # =========================
    headers = {

        "Referer":
            driver.current_url,

        "User-Agent":
            driver.execute_script(
                "return navigator.userAgent;"
            ),

        "X-Requested-With":
            "XMLHttpRequest"
    }

    # =========================
    # SEND REQUEST
    # =========================
    try:

        response = session.post(
            update_url,
            data=payload,
            headers=headers,
            allow_redirects=True
        )

        print(
            "✅ Request update terkirim"
        )

    except Exception as e:

        print(
            "❌ Gagal kirim request"
        )

        print(e)

        return

    # =========================
    # DEBUG RESPONSE
    # =========================
    print("========== RESPONSE ==========")

    print(
        "STATUS CODE:",
        response.status_code
    )

    print(
        "FINAL URL:",
        response.url
    )

    print("==============================")

    # =========================
    # VALIDASI RESULT
    # =========================
    if "/login" in response.url:

        print(
            "❌ Redirect ke LOGIN"
        )

        print(
            "⚠️ Session/auth middleware "
            "masih ditolak"
        )

        with open(
            "response_login.html",
            "w",
            encoding="utf-8"
        ) as f:

            f.write(response.text)

        print(
            "✅ HTML response disimpan "
            "ke response_login.html"
        )

        return

    elif response.status_code == 200:


        print("✅✅✅ Skenario Tambah Supplier Berhasil dengan waktu " + ' ' + str(time.time()) + ' Detik')

        return

    else:

        print(
            "⚠️ Response tidak dikenali"
        )

        print(response.text[:1000])

        return


test_add_suppliers_item(driver)

✅ Klik tombol suppliers berhasil
Masukkan Nama Supplier: PT Ompreng
Masukkan Kontak yang bisa dihubungi: Dzul
Masukkan Nomor Telepon yang bisa dihubungi: 098778987890
Masukkan Email yang bisa dihubungi: dzul@gmail.com
Masukkan Alamat yang bisa dirujuk: Jalanin
Masukkan Kota Anda: Sidoarjo
✅ CSRF TOKEN:
XD0oYw5JKRCp1qMG18cKYzJLOJmJWOoM0pjTWWBs
✅ Cookies berhasil dipindahkan
UPDATE URL:
https://larynfood.com/suppliers
✅ Request update terkirim
========== RESPONSE ==========
STATUS CODE: 200
FINAL URL: https://larynfood.com/suppliers
✅✅✅ Skenario Tambah Supplier Berhasil dengan waktu  1778380619.8992314 Detik


**Edit Supplier**

In [ ]:
def test_edit_suppliers_with_id(driver):
    driver.get("https://larynfood.com/suppliers")

    names = input("Masukkan Supplier yang ingin diedit: ")

    soup = BeautifulSoup(driver.page_source, 'lxml')
    table = soup.find('table', class_='table')



    rows = table.find_all('tr')[1:]

    for row in rows:
        cols = row.find_all('td')
        if not cols: continue

        supplier_id = None

        nama = cols[0].get_text(strip=True)

        if nama == names:
            print(f"Match ditemukan: {nama}")
            link_edit = cols[5].find('a', href=re.compile(r'/suppliers/\d+/edit'))

            suppliers_id = None
            if link_edit:
                href = link_edit.get('href')

                match = re.search(r'/suppliers/(\d+)/edit', href)
                if match:
                    suppliers_id = match.group(1)

            xpath_edit = f"//td[text()='{names}']/parent::tr//a[starts-with(@href, 'https://larynfood.com/suppliers/{suppliers_id}/edit')]"


            edit_button = driver.find_element(By.XPATH, xpath_edit)

            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", edit_button)
            time.sleep(0.5)

            edit_button.click()

            print("✅ Berhasil masuk halaman edit suppliers")

            supplier = input("Masukkan Nama Supplier: ")
            supplier_input = "input[name='nama_supplier']"

            kontak = input("Masukkan Kontak yang bisa dihubungi: ")
            kontak_input = "input[name='kontak']"

            nomor_telepon = input("Masukkan Nomor Telepon yang bisa dihubungi: ")
            telepon_input = "input[name='nomor_telepon']"

            email = input("Masukkan Email yang bisa dihubungi: ")
            email_input = "input[name='email']"

            alamat = input("Masukkan Alamat yang bisa dirujuk: ")
            alamat_input = "textarea[name='alamat']"

            kota = input("Masukkan Kota Anda: ")
            alamat_input = "input[name='kota']"

            data_supplier = {
            supplier: supplier_input,
            kontak: kontak_input,
            nomor_telepon: telepon_input,
            email: email_input,
            alamat: alamat_input,
            kota: alamat_input
            }


            # Jika ingin dimasukkan ke dalam satu Array (List)
            daftar_supplier = [data_supplier]

            for data in daftar_supplier:
                for nama, input_field in data.items():
                    driver.find_element(By.CSS_SELECTOR, input_field).send_keys(nama)


            session = requests.Session()


            # =========================
            # AMBIL CSRF TOKEN
            # =========================
            try:

                csrf_token = (
                    driver.find_element(
                        By.NAME,
                        "_token"
                    ).get_attribute("value")
                )

                print("✅ CSRF TOKEN:")
                print(csrf_token)

            except Exception as e:

                print(
                    "❌ Gagal ambil CSRF token"
                )

                print(e)

                return

            # =========================
            # COPY COOKIES SELENIUM
            # =========================
            selenium_cookies = (
                driver.get_cookies()
            )

            for cookie in selenium_cookies:

                session.cookies.set(
                    cookie["name"],
                    cookie["value"]
                )

            print(
                "✅ Cookies berhasil dipindahkan"
            )

            # =========================
            # UPDATE URL
            # =========================
            update_url = (
                f"https://larynfood.com/"
                f"suppliers/{suppliers_id}"
            )

            print("UPDATE URL:")
            print(update_url)

            payload = {

                "_token": csrf_token,

                "_method": "PUT",

                "nama_supplier": supplier,

                "kontak": kontak,

                "nomor_telepon": nomor_telepon,

                "email": email,

                "alamat": alamat,

                "kota": kota
            }

            # =========================
            # HEADERS
            # =========================
            headers = {

                "Referer":
                    driver.current_url,

                "User-Agent":
                    driver.execute_script(
                        "return navigator.userAgent;"
                    ),

                "X-Requested-With":
                    "XMLHttpRequest"
            }

            # =========================
            # SEND REQUEST
            # =========================
            try:

                response = session.post(
                    update_url,
                    data=payload,
                    headers=headers,
                    allow_redirects=True
                )

                print(
                    "✅ Request update terkirim"
                )

            except Exception as e:

                print(
                    "❌ Gagal kirim request"
                )

                print(e)

                return

            # =========================
            # DEBUG RESPONSE
            # =========================
            print("========== RESPONSE ==========")

            print(
                "STATUS CODE:",
                response.status_code
            )

            print(
                "FINAL URL:",
                response.url
            )

            print("==============================")

            # =========================
            # VALIDASI RESULT
            # =========================
            if "/login" in response.url:

                print(
                    "❌ Redirect ke LOGIN"
                )

                print(
                    "⚠️ Session/auth middleware "
                    "masih ditolak"
                )

                with open(
                    "response_login.html",
                    "w",
                    encoding="utf-8"
                ) as f:

                    f.write(response.text)

                print(
                    "✅ HTML response disimpan "
                    "ke response_login.html"
                )

                return

            elif response.status_code == 200:

                print("✅✅✅ Skenario Edit Supplier Berhasil dengan waktu " + ' ' + str(time.time()) + ' Detik')

                return

            else:

                print(
                    "⚠️ Response tidak dikenali"
                )

                print(response.text[:1000])

                return


test_edit_suppliers_with_id(driver)

Masukkan Supplier yang ingin diedit: PT Ompreng
Match ditemukan: PT Ompreng
✅ Berhasil masuk halaman edit suppliers
Masukkan Nama Supplier: PT Ompreng Perkasa
Masukkan Kontak yang bisa dihubungi: Dzul
Masukkan Nomor Telepon yang bisa dihubungi: 078987897890
Masukkan Email yang bisa dihubungi: dzul@gmail.com
Masukkan Alamat yang bisa dirujuk: Jalanin aja
Masukkan Kota Anda: Surabaya
✅ CSRF TOKEN:
XD0oYw5JKRCp1qMG18cKYzJLOJmJWOoM0pjTWWBs
✅ Cookies berhasil dipindahkan
UPDATE URL:
https://larynfood.com/suppliers/12
✅ Request update terkirim
========== RESPONSE ==========
STATUS CODE: 200
FINAL URL: https://larynfood.com/suppliers
✅✅✅ Skenario Edit Supplier Berhasil dengan waktu  1778380659.734652 Detik


**Hapus Supplier**

In [ ]:
def test_delete_suppliers_with_id(driver):
    driver.get("https://larynfood.com/suppliers")

    names = input("Masukkan suppliers yang ingin dihapus: ")
    found = False

    while True:
        soup = BeautifulSoup(driver.page_source, 'lxml')
        table = soup.find('table', class_='table')

        if not table: break

        rows = table.find_all('tr')[1:]

        for row in rows:
            cols = row.find_all('td')
            if not cols: continue

            nama = cols[0].get_text(strip=True)

            if nama == names:
                print(f"Match ditemukan: {nama}")

                xpath_delete = f"//td[text()='{names}']/parent::tr//button[@type='submit']"
                delete_button = driver.find_element(By.XPATH, xpath_delete)

                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", delete_button)
                time.sleep(0.5)

                delete_button.click()

                try:
                    WebDriverWait(driver, 5).until(EC.alert_is_present())
                    alert = driver.switch_to.alert
                    print(f"Isi Alert: {alert.text}")
                    message_input = input('Ok atau Cancel ')
                    if message_input == 'Ok':
                      alert.accept() # Klik OK
                    else:
                      alert.dismiss() # Klik Cancel
                    print(f"🗑️ Supplier '{names}' berhasil dihapus.")
                    found = True
                except:
                    print("Tidak ada alert konfirmasi.")

                time.sleep(2)
                print("✅✅✅ Skenario Hapus Suppliers Berhasil dengan waktu" + ' ' + str(time.time()) + ' Detik')
                return

        # Logika Pagination: Jika tidak ketemu di halaman ini, coba klik 'Next'
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")
            next_button.click()
            time.sleep(2) # Tunggu loading halaman berikutnya
        except:
            print("Nama tidak ditemukan hingga halaman terakhir.")
            break

test_delete_suppliers_with_id(driver)


Masukkan suppliers yang ingin dihapus: PT Ompreng
Match ditemukan: PT Ompreng
Isi Alert: Yakin ingin menghapus?
Ok atau Cancel Ok
🗑️ Supplier 'PT Ompreng' berhasil dihapus.
✅✅✅ Skenario Hapus Suppliers Berhasil dengan waktu 1778381075.4617617 Detik


# **FITUR STOCK GUDANG**

**Daftar Stock Gudang**

In [ ]:
import pandas as pd
import re # Import Regex untuk ambil angka dari URL

time.sleep(5)

def test_list_stock_with_id(driver):
    driver.get("https://larynfood.com/stock-gudang")
    all_data = []

    while True:
        soup = BeautifulSoup(driver.page_source, 'lxml')
        table = soup.find('table', class_='table')

        if not table: break

        rows = table.find_all('tr')[1:] # Lewati header

        for row in rows:
            cols = row.find_all('td')
            if cols:

                sku = cols[0].get_text(strip=True)
                nama_produk = cols[1].get_text(strip=True)
                kategori = cols[2].get_text(strip=True)
                stok = cols[3].get_text(strip=True)
                satuan = cols[4].get_text(strip=True)
                harga = cols[5].get_text(strip=True)

                link_edit = cols[7].find('a', href=re.compile(r'/stock-gudang/\d+'))

                stock_id = None
                if link_edit:
                    href = link_edit.get('href')

                    match = re.search(r'/stock-gudang/(\d+)', href)
                    if match:
                        stock_id = match.group(1)

                all_data.append([stock_id, nama_produk, kategori, stok, satuan, harga])

        for _ in range(10):
            driver.execute_script("window.scrollBy(0, 500);")
            time.sleep(0.3)

        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")

            driver.execute_script("arguments[0].scrollIntoView();", next_button)
            next_button.click()
        except:

            print("Selesai mengambil semua halaman.")
            break


    df = pd.DataFrame(all_data, columns=["ID", "Nama Produk", "Kategori", "Stok", "Satuan", "Harga"])
    return df

test_list_stock_with_id(driver)

Selesai mengambil semua halaman.


,ID,Nama Produk,Kategori,Stok,Satuan,Harga
0,13,Dimsum frozen,-,3 pack + 72 pcs(Total: pcs),pcs,Rp 60.000
1,12,dimsum,Dimsum kukus,0 pack + 12 pcs(Total: pcs),BOX,Rp 0


**Tambah Stok Gudang**

In [ ]:
from selenium.webdriver.support.ui import Select

def test_add_stock_item(driver):

    driver.get("https://larynfood.com/stock-gudang")

    clickTombolStock = "a[href^='https://larynfood.com/stock-gudang/create']"
    wait = WebDriverWait(driver, 10)

    # Tunggu sampai elemen ada di DOM
    element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, clickTombolStock)))

    # Klik menggunakan JavaScript
    driver.execute_script("arguments[0].click();", element)
    print("✅ Klik tombol Stock Gudang berhasil")

    sku = input("Masukkan SKU: ")
    sku_input = "input[name='sku']"

    satuan = input("Masukkan Satuan Produk: ")
    satuan_input = "input[name='satuan']"

    konversi_satuan = input("Masukkan PCS Produk: ")
    konversiSatuan_input = "input[name='konversi_satuan']"

    nama_produk = input("Masukkan Nama Produk yang ingin dijual: ")
    namaProduk_input = "input[name='nama_produk']"

    lokasi_gudang = input("Masukkan Alamat yang bisa dirujuk: ")
    lokasiGudang_input = "input[name='lokasi_gudang']"

    data_stock = {
    sku : sku_input,
    satuan : satuan_input,
    konversi_satuan : konversiSatuan_input,
    nama_produk : namaProduk_input,
    lokasi_gudang : lokasiGudang_input
    }


    # Jika ingin dimasukkan ke dalam satu Array (List)
    daftar_stock = [data_stock]

    for data in daftar_stock:
        for nama, input_field in data.items():
            driver.find_element(By.CSS_SELECTOR, input_field).send_keys(nama)



    dropdown = Select(driver.find_element(By.NAME, "category_id"))

    # Tampilkan semua pilihan ke terminal
    print("\n--- Daftar Kategori ---")
    for index, option in enumerate(dropdown.options):
        print(f"{index + 1}. {option.text}")

    # Minta input berdasarkan nomor yang tampil
    pilih = int(input("\nMasukkan nomor pilihan Anda: "))

    if 0 < pilih <= len(dropdown.options):
      dropdown.options[pilih - 1].click()
    else:
      print("Pilihan di luar jangkauan!")

    dropdown2 = Select(driver.find_element(By.NAME, "supplier_id"))

    # Tampilkan semua pilihan ke terminal
    print("\n--- Daftar Supplier ---")
    for index, option in enumerate(dropdown2.options):
        print(f"{index + 1}. {option.text}")

    # Minta input berdasarkan nomor yang tampil
    pilih2 = int(input("\nMasukkan nomor pilihan Anda: "))

    if 0 < pilih2 <= len(dropdown2.options):
      dropdown2.options[pilih2 - 1].click()
    else:
      print("Pilihan di luar jangkauan!")


    session = requests.Session()


    # =========================
    # AMBIL CSRF TOKEN
    # =========================
    try:

        csrf_token = (
            driver.find_element(
                By.NAME,
                "_token"
            ).get_attribute("value")
        )

        print("✅ CSRF TOKEN:")
        print(csrf_token)

    except Exception as e:

        print(
            "❌ Gagal ambil CSRF token"
        )

        print(e)

        return

    # =========================
    # COPY COOKIES SELENIUM
    # =========================
    selenium_cookies = (
        driver.get_cookies()
    )

    for cookie in selenium_cookies:

        session.cookies.set(
            cookie["name"],
            cookie["value"]
        )

    print(
        "✅ Cookies berhasil dipindahkan"
    )

    # =========================
    # UPDATE URL
    # =========================
    update_url = (
        f"https://larynfood.com/"
        f"stock-gudang"
    )

    print("UPDATE URL:")
    print(update_url)

    payload = {

        "_token": csrf_token,

        "_method": "POST",

        "sku": sku,

        "satuan": satuan,

        "konversi_satuan": konversi_satuan,

        "nama_produk": nama_produk,

        "lokasi_gudang": lokasi_gudang,

        "category_id": pilih,

        "supplier_id": pilih2

    }

    # =========================
    # HEADERS
    # =========================
    headers = {

        "Referer":
            driver.current_url,

        "User-Agent":
            driver.execute_script(
                "return navigator.userAgent;"
            ),

        "X-Requested-With":
            "XMLHttpRequest"
    }

    # =========================
    # SEND REQUEST
    # =========================
    try:

        response = session.post(
            update_url,
            data=payload,
            headers=headers,
            allow_redirects=True
        )

        print(
            "✅ Request update terkirim"
        )

    except Exception as e:

        print(
            "❌ Gagal kirim request"
        )

        print(e)

        return

    # =========================
    # DEBUG RESPONSE
    # =========================
    print("========== RESPONSE ==========")

    print(
        "STATUS CODE:",
        response.status_code
    )

    print(
        "FINAL URL:",
        response.url
    )

    print("==============================")

    # =========================
    # VALIDASI RESULT
    # =========================
    if "/login" in response.url:

        print(
            "❌ Redirect ke LOGIN"
        )

        print(
            "⚠️ Session/auth middleware "
            "masih ditolak"
        )

        with open(
            "response_login.html",
            "w",
            encoding="utf-8"
        ) as f:

            f.write(response.text)

        print(
            "✅ HTML response disimpan "
            "ke response_login.html"
        )

        return

    elif response.status_code == 200:


        print("✅✅✅ Skenario Tambah Stock Gudang Berhasil dengan waktu " + ' ' + str(time.time()) + ' Detik')

        return

    else:

        print(
            "⚠️ Response tidak dikenali"
        )

        print(response.text[:1000])

        return


test_add_stock_item(driver)

✅ Klik tombol Stock Gudang berhasil
Masukkan SKU: PKT-300
Masukkan Satuan Produk: PCS
Masukkan PCS Produk: 20
Masukkan Nama Produk yang ingin dijual: Dimsum Mini Crot
Masukkan Alamat yang bisa dirujuk: Sidoarjo

--- Daftar Kategori ---
1. -- Pilih Kategori --
2. Dimsum kukus
3. Saos Mentai
4. Chili oil
5. Saos Bangkok
6. Parseley
7. Keju
8. Garnish (hiasan makanan)
9. Dimsum Goreng
10. Udang keju
11. Kemasan Tray Aluminium
12. Kemasan Box
13. Sumpit
14. Kemasan kresek
15. Kresek plastik
16. Tutup Tray aluminium
17. Biji Wijen

Masukkan nomor pilihan Anda: 3

--- Daftar Supplier ---
1. -- Pilih Supplier --
2. YOIKI Frozen Food
3. Toko Shopee
4. Toko pasar sepanjang
5. Toko Nico
6. Nisrina Foods
7. INCLUDED

Masukkan nomor pilihan Anda: 2
✅ CSRF TOKEN:
XD0oYw5JKRCp1qMG18cKYzJLOJmJWOoM0pjTWWBs
✅ Cookies berhasil dipindahkan
UPDATE URL:
https://larynfood.com/stock-gudang
✅ Request update terkirim
========== RESPONSE ==========
STATUS CODE: 200
FINAL URL: https://larynfood.com/stock-gudang


**Edit Stok Gudang**

In [ ]:
from selenium.webdriver.support.ui import Select
def test_edit_stock_with_id(driver):
    driver.get("https://larynfood.com/stock-gudang")

    names = input("Masukkan Stock yang ingin diedit: ")

    soup = BeautifulSoup(driver.page_source, 'lxml')
    table = soup.find('table', class_='table')



    rows = table.find_all('tr')[1:]

    for row in rows:
        cols = row.find_all('td')
        if not cols: continue

        stock_id = None

        nama = cols[1].get_text(strip=True)

        if nama == names:
            print(f"Match ditemukan: {nama}")
            link_edit = cols[7].find('a', href=re.compile(r'/stock-gudang/\d+/edit'))

            stock_id = None
            if link_edit:
                href = link_edit.get('href')

                match = re.search(r'/stock-gudang/(\d+)/edit', href)
                if match:
                    stock_id = match.group(1)

            xpath_edit = f"//td[text()='{names}']/parent::tr//a[starts-with(@href, 'https://larynfood.com/stock-gudang/{stock_id}/edit')]"


            edit_button = driver.find_element(By.XPATH, xpath_edit)

            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", edit_button)
            time.sleep(0.5)

            edit_button.click()

            print("✅ Berhasil masuk halaman edit stock")

            sku = input("Masukkan SKU: ")
            sku_input = "input[name='sku']"

            satuan = input("Masukkan Satuan Produk: ")
            satuan_input = "input[name='satuan']"

            konversi_satuan = input("Masukkan PCS Produk: ")
            konversiSatuan_input = "input[name='konversi_satuan']"

            nama_produk = input("Masukkan Nama Produk yang ingin dijual: ")
            namaProduk_input = "input[name='nama_produk']"

            jml_pack = input("Masukkan Jumlah pack yang ingin dijual: ")
            jmlPack_input = "input[name='jumlah_pack']"

            data_stock = {
            sku : sku_input,
            satuan : satuan_input,
            konversi_satuan : konversiSatuan_input,
            nama_produk : namaProduk_input,
            jml_pack : jmlPack_input
            }


            # Jika ingin dimasukkan ke dalam satu Array (List)
            daftar_stock = [data_stock]

            for data in daftar_stock:
                for nama, input_field in data.items():
                    driver.find_element(By.CSS_SELECTOR, input_field).send_keys(nama)



            dropdown = Select(driver.find_element(By.NAME, "category_id"))

            # Tampilkan semua pilihan ke terminal
            print("\n--- Daftar Kategori ---")
            for index, option in enumerate(dropdown.options):
                print(f"{index + 1}. {option.text}")

            # Minta input berdasarkan nomor yang tampil
            pilih = int(input("\nMasukkan nomor pilihan Anda: "))

            if 0 < pilih <= len(dropdown.options):
              dropdown.options[pilih - 1].click()
            else:
              print("Pilihan di luar jangkauan!")

            dropdown2 = Select(driver.find_element(By.NAME, "supplier_id"))

            # Tampilkan semua pilihan ke terminal
            print("\n--- Daftar Supplier ---")
            for index, option in enumerate(dropdown2.options):
                print(f"{index + 1}. {option.text}")

            # Minta input berdasarkan nomor yang tampil
            pilih2 = int(input("\nMasukkan nomor pilihan Anda: "))

            if 0 < pilih2 <= len(dropdown2.options):
              dropdown2.options[pilih2 - 1].click()
            else:
              print("Pilihan di luar jangkauan!")


            session = requests.Session()


            # =========================
            # AMBIL CSRF TOKEN
            # =========================
            try:

                csrf_token = (
                    driver.find_element(
                        By.NAME,
                        "_token"
                    ).get_attribute("value")
                )

                print("✅ CSRF TOKEN:")
                print(csrf_token)

            except Exception as e:

                print(
                    "❌ Gagal ambil CSRF token"
                )

                print(e)

                return

            # =========================
            # COPY COOKIES SELENIUM
            # =========================
            selenium_cookies = (
                driver.get_cookies()
            )

            for cookie in selenium_cookies:

                session.cookies.set(
                    cookie["name"],
                    cookie["value"]
                )

            print(
                "✅ Cookies berhasil dipindahkan"
            )

            # =========================
            # UPDATE URL
            # =========================
            update_url = (
                f"https://larynfood.com/"
                f"stock-gudang/{stock_id}"
            )

            print("UPDATE URL:")
            print(update_url)

            payload = {

                "_token": csrf_token,

                "_method": "PUT",

                "sku": sku,

                "satuan": satuan,

                "konversi_satuan": konversi_satuan,

                "nama_produk": nama_produk,

                "jumlah_pack": jml_pack,

                "category_id": pilih,

                "supplier_id": pilih2
            }

            # =========================
            # HEADERS
            # =========================
            headers = {

                "Referer":
                    driver.current_url,

                "User-Agent":
                    driver.execute_script(
                        "return navigator.userAgent;"
                    ),

                "X-Requested-With":
                    "XMLHttpRequest"
            }

            # =========================
            # SEND REQUEST
            # =========================
            try:

                response = session.post(
                    update_url,
                    data=payload,
                    headers=headers,
                    allow_redirects=True
                )

                print(
                    "✅ Request update terkirim"
                )

            except Exception as e:

                print(
                    "❌ Gagal kirim request"
                )

                print(e)

                return

            # =========================
            # DEBUG RESPONSE
            # =========================
            print("========== RESPONSE ==========")

            print(
                "STATUS CODE:",
                response.status_code
            )

            print(
                "FINAL URL:",
                response.url
            )

            print("==============================")

            # =========================
            # VALIDASI RESULT
            # =========================
            if "/login" in response.url:

                print(
                    "❌ Redirect ke LOGIN"
                )

                print(
                    "⚠️ Session/auth middleware "
                    "masih ditolak"
                )

                with open(
                    "response_login.html",
                    "w",
                    encoding="utf-8"
                ) as f:

                    f.write(response.text)

                print(
                    "✅ HTML response disimpan "
                    "ke response_login.html"
                )

                return

            elif response.status_code == 200:

                print("✅✅✅ Skenario Edit Stock Gudang Berhasil dengan waktu " + ' ' + str(time.time()) + ' Detik')

                return

            else:

                print(
                    "⚠️ Response tidak dikenali"
                )

                print(response.text[:1000])

                return


test_edit_stock_with_id(driver)

Masukkan Stock yang ingin diedit: Dimsum Mini Crot
Match ditemukan: Dimsum Mini Crot
✅ Berhasil masuk halaman edit stock
Masukkan SKU: PKT-400
Masukkan Satuan Produk: PCS
Masukkan PCS Produk: 20
Masukkan Nama Produk yang ingin dijual: Dimsum Crot Enak
Masukkan Jumlah pack yang ingin dijual: 23

--- Daftar Kategori ---
1. Pilih Kategori
2. Dimsum kukus
3. Saos Mentai
4. Chili oil
5. Saos Bangkok
6. Parseley
7. Keju
8. Garnish (hiasan makanan)
9. Dimsum Goreng
10. Udang keju
11. Kemasan Tray Aluminium
12. Kemasan Box
13. Sumpit
14. Kemasan kresek
15. Kresek plastik
16. Tutup Tray aluminium
17. Biji Wijen

Masukkan nomor pilihan Anda: 3

--- Daftar Supplier ---
1. Pilih Supplier
2. YOIKI Frozen Food
3. Toko Shopee
4. Toko pasar sepanjang
5. Toko Nico
6. Nisrina Foods
7. INCLUDED

Masukkan nomor pilihan Anda: 2
✅ CSRF TOKEN:
XD0oYw5JKRCp1qMG18cKYzJLOJmJWOoM0pjTWWBs
✅ Cookies berhasil dipindahkan
UPDATE URL:
https://larynfood.com/stock-gudang/15
✅ Request update terkirim
========== RESPONSE

**Hapus Stok Gudang**

In [ ]:
def test_delete_stock_with_id(driver):
    driver.get("https://larynfood.com/stock-gudang")

    names = input("Masukkan Stock yang ingin dihapus: ")
    found = False

    while True:
        soup = BeautifulSoup(driver.page_source, 'lxml')
        table = soup.find('table', class_='table')

        if not table: break

        rows = table.find_all('tr')[1:]

        for row in rows:
            cols = row.find_all('td')
            if not cols: continue

            nama = cols[1].get_text(strip=True)

            if nama == names:
                print(f"Match ditemukan: {nama}")

                xpath_delete = f"//td[text()='{names}']/parent::tr//button[@type='submit']"
                delete_button = driver.find_element(By.XPATH, xpath_delete)

                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", delete_button)
                time.sleep(0.5)

                delete_button.click()

                try:
                    WebDriverWait(driver, 5).until(EC.alert_is_present())
                    alert = driver.switch_to.alert
                    print(f"Isi Alert: {alert.text}")
                    message_input = input('Ok atau Cancel ')
                    if message_input == 'Ok':
                      alert.accept() # Klik OK
                    else:
                      alert.dismiss() # Klik Cancel
                    print(f"🗑️ Supplier '{names}' berhasil dihapus.")
                    found = True
                except:
                    print("Tidak ada alert konfirmasi.")

                time.sleep(2)
                print("✅✅✅ Skenario Hapus Stock Gudang Berhasil dengan waktu" + ' ' + str(time.time()) + ' Detik')
                return

        # Logika Pagination: Jika tidak ketemu di halaman ini, coba klik 'Next'
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")
            next_button.click()
            time.sleep(2) # Tunggu loading halaman berikutnya
        except:
            print("Nama tidak ditemukan hingga halaman terakhir.")
            break

test_delete_stock_with_id(driver)


Masukkan Stock yang ingin dihapus: Dimsum Mini Crot
Match ditemukan: Dimsum Mini Crot
Isi Alert: Yakin ingin menghapus stock ini?
Ok atau Cancel Ok
🗑️ Supplier 'Dimsum Mini Crot' berhasil dihapus.
✅✅✅ Skenario Hapus Stock Gudang Berhasil dengan waktu 1778381050.9546502 Detik


# **FITUR PRODUK PAKET**

**Daftar Produk Paket**

In [ ]:
import pandas as pd
import re # Import Regex untuk ambil angka dari URL

time.sleep(5)

def test_list_stock_with_id(driver):
    driver.get("https://larynfood.com/produk-paket")
    all_data = []

    while True:
        soup = BeautifulSoup(driver.page_source, 'lxml')
        table = soup.find('table', class_='table')

        if not table: break

        rows = table.find_all('tr')[1:] # Lewati header

        for row in rows:
            cols = row.find_all('td')
            if cols:

                nama_paket = cols[0].get_text(strip=True)
                kode = cols[1].get_text(strip=True)
                jml_item = cols[2].get_text(strip=True)
                hpp_total = cols[3].get_text(strip=True)
                produk_siap = cols[4].get_text(strip=True)
                status = cols[5].get_text(strip=True)

                link_edit = cols[6].find('a', href=re.compile(r'/produk-paket/\d+'))

                paket_id = None
                if link_edit:
                    href = link_edit.get('href')

                    match = re.search(r'/produk-paket/(\d+)', href)
                    if match:
                        paket_id = match.group(1)

                all_data.append([paket_id, nama_paket, kode, jml_item, hpp_total, produk_siap, status])

        for _ in range(10):
            driver.execute_script("window.scrollBy(0, 500);")
            time.sleep(0.3)

        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")

            driver.execute_script("arguments[0].scrollIntoView();", next_button)
            next_button.click()
        except:

            print("Selesai mengambil semua halaman.")
            break


    df = pd.DataFrame(all_data, columns=["ID", "Nama Paket", "Kode", "Jumlah Item", "HPP Total" ,"Produk Siap Jual", "Status"])
    return df

test_list_stock_with_id(driver)

Selesai mengambil semua halaman.


,ID,Nama Paket,Kode,Jumlah Item,HPP Total,Produk Siap Jual,Status
0,2,Bikin dumsum,-,1 item,Rp 0,0 paket,Aktif


**Tambah Produk Paket**

In [ ]:
from selenium.webdriver.support.ui import Select

def test_add_packet_item(driver):

    driver.get("https://larynfood.com/produk-paket")

    clickTombolPacket = "a[href^='https://larynfood.com/produk-paket/create']"
    wait = WebDriverWait(driver, 10)

    element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, clickTombolPacket)))

    # Klik menggunakan JavaScript
    driver.execute_script("arguments[0].click();", element)
    print("✅ Klik tombol Paket berhasil")

    nama_paket = input("Masukkan Nama Paket: ")
    namaPaket_input = "input[name='nama_paket']"

    kode_paket = input("Masukkan Kode Paket: ")
    kodePaket_input = "input[name='kode_paket']"


    deskripsi = input("Masukkan Deskripsi Paket: ")
    deskripsi_input = "textarea[name='deskripsi']"

    data_paket = {
    nama_paket : namaPaket_input,
    kode_paket : kodePaket_input,
    deskripsi : deskripsi_input
    }


    # Jika ingin dimasukkan ke dalam satu Array (List)
    daftar_paket = [data_paket]

    for data in daftar_paket:
        for nama, input_field in data.items():
            driver.find_element(By.CSS_SELECTOR, input_field).send_keys(nama)



    dropdown = Select(driver.find_element(By.NAME, "status"))

    # Tampilkan semua pilihan ke terminal
    print("\n--- Daftar Status ---")
    for index, option in enumerate(dropdown.options):
        print(f"{index + 1}. {option.text}")

    # Minta input berdasarkan nomor yang tampil
    pilihan = int(input("\nMasukkan nomor pilihan Anda: "))

    pilih = "valid"

    if 0 < pilihan <= len(dropdown.options):
      dropdown.options[pilihan - 1].click()
      value_terpilih = dropdown.options[pilihan - 1].get_attribute("value")

      # Semua pilihan yang ada di dalam dropdown adalah VALID secara UI
      pilih = value_terpilih
      print(f"✅ Opsi terpilih: {value_terpilih} (Status: {value_terpilih})")
    else:
      print("Pilihan di luar jangkauan!")


    clickTombolItem = "button[id='btnTambahItem']"
    element2 = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, clickTombolItem)))

    # Klik menggunakan JavaScript
    driver.execute_script("arguments[0].click();", element2)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table[id='tableItems']")))
    print("✅ Table Item Berhasil tampil")

    time.sleep(1)

    tombol_tambah = element2

    jml_item = int(input("\nMasukkan Jumlah Item: "))

    items_payload = []
    for i in range(jml_item):
        # 1. Klik tombol tambah (pastikan tombol_tambah didefinisikan dengan benar)
        driver.execute_script("arguments[0].click();", tombol_tambah)
        print(f"➕ Menambah baris ke-{i}...")

        selector_check = f"select[name='items[{i}][stock_gudang_id]']"
        dropdown_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector_check)))

        select_item = Select(dropdown_element)

        print(f"\n--- Daftar Item {i} ---")
        for index, option in enumerate(select_item.options):
            print(f"{index + 1}. {option.text}")

        pilih2 = int(input("Masukkan nomor pilihan Anda: "))

        if 0 < pilih2 <= len(select_item.options):
            # FIX 1: Gunakan pilih2, bukan pilihan
            value_terpilih2 = select_item.options[pilih2 - 1].get_attribute("value")

            # Pilih di UI agar sinkron
            select_item.select_by_value(value_terpilih2)

            # AMBIL DATA-HPP DARI ATRIBUT HTML
            hpp_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, f"tr:nth-child({i+1}) .hpp-pcs-display")))

            # Gunakan .text (bukan getText)
            hpp_text = hpp_element.text or "0"
        else:
            print("Pilihan di luar jangkauan!")
            continue

        qty_input = driver.find_element(By.CSS_SELECTOR, f"input[name='items[{i}][qty_per_paket]']")
        qty_input.clear()
        qty = int(input("Masukkan Quantity: "))
        qty_input.send_keys(qty)
        driver.execute_script("arguments[0].dispatchEvent(new Event('input'));", qty_input)

        hpp_satuan = hpp_text.replace("Rp", "").replace(".", "").replace(",", "").strip()


        try:
            hpp_satuan_int = int(hpp_satuan)
        except ValueError:
            hpp_satuan_int = 0



        subtotal_item = hpp_satuan_int * qty

        item_baru = {
            "stock_gudang_id": value_terpilih2,
            "qty_per_paket": qty,
            "hpp_satuan": hpp_satuan_int,
            "subtotal_hpp": subtotal_item
        }

        items_payload.append(item_baru)

    session = requests.Session()


    # =========================
    # AMBIL CSRF TOKEN
    # =========================
    try:

        csrf_token = (
            driver.find_element(
                By.NAME,
                "_token"
            ).get_attribute("value")
        )

        print("✅ CSRF TOKEN:")
        print(csrf_token)

    except Exception as e:

        print(
            "❌ Gagal ambil CSRF token"
        )

        print(e)

        return

    # =========================
    # COPY COOKIES SELENIUM
    # =========================
    selenium_cookies = (
        driver.get_cookies()
    )

    for cookie in selenium_cookies:

        session.cookies.set(
            cookie["name"],
            cookie["value"]
        )

    print(
        "✅ Cookies berhasil dipindahkan"
    )

    # =========================
    # UPDATE URL
    # =========================
    update_url = (
        f"https://larynfood.com/"
        f"produk-paket"
    )

    print("UPDATE URL:")
    print(update_url)

    payload = {
        "_token": csrf_token,
        "_method": "POST",
        "nama_paket": nama_paket,
        "kode_paket": kode_paket,
        "deskripsi": deskripsi,
        "status": pilih, # Pastikan berisi "aktif" atau "nonaktif"
        "items": items_payload,     # Kita gunakan list/array untuk menampung item
    }


    # =========================
    # HEADERS
    # =========================
    headers = {

        "Referer":
            driver.current_url,

        "User-Agent":
            driver.execute_script(
                "return navigator.userAgent;"
            ),

        "X-Requested-With":
            "XMLHttpRequest",

        "Accept": "application/json", # WAJIB agar response berbentuk JSON
    "Content-Type": "application/json"
    }

    # =========================
    # SEND REQUEST
    # =========================
    try:

        response = session.post(
            update_url,
            json=payload,
            headers=headers,
            allow_redirects=True
        )

        print(
            "✅ Request update terkirim"
        )

    except Exception as e:

        print(
            "❌ Gagal kirim request"
        )

        print(e)

        return

    # =========================
    # DEBUG RESPONSE
    # =========================
    print("========== RESPONSE ==========")

    print(
        "STATUS CODE:",
        response.status_code
    )

    print(
        "FINAL URL:",
        response.url
    )

    print("==============================")

    # =========================
    # VALIDASI RESULT
    # =========================
    if "/login" in response.url:

        print(
            "❌ Redirect ke LOGIN"
        )

        print(
            "⚠️ Session/auth middleware "
            "masih ditolak"
        )

        with open(
            "response_login.html",
            "w",
            encoding="utf-8"
        ) as f:

            f.write(response.text)

        print(
            "✅ HTML response disimpan "
            "ke response_login.html"
        )

        return

    elif response.status_code == 200:


        print("✅✅✅ Skenario Tambah Produk Paket Berhasil dengan waktu " + ' ' + str(time.time()) + ' Detik')

        return

    else:

        print(
            "⚠️ Response tidak dikenali"
        )

        print(response.text[:1000])

        return


test_add_packet_item(driver)

✅ Klik tombol Paket berhasil
Masukkan Nama Paket: Paket Hemat
Masukkan Kode Paket: PKT-4005
Masukkan Deskripsi Paket: -

--- Daftar Status ---
1. Aktif
2. Nonaktif

Masukkan nomor pilihan Anda: 2
✅ Opsi terpilih: nonaktif (Status: nonaktif)
✅ Table Item Berhasil tampil

Masukkan Jumlah Item: 2
➕ Menambah baris ke-0...

--- Daftar Item 0 ---
1. -- Pilih Item Stock Gudang --
2. dimsum (12 PCS) - Dimsum kukus
3. Dimsum frozen (72 PCS) - Tanpa Kategori
Masukkan nomor pilihan Anda: 2
Masukkan Quantity: 12
➕ Menambah baris ke-1...

--- Daftar Item 1 ---
1. -- Pilih Item Stock Gudang --
2. dimsum (12 PCS) - Dimsum kukus
3. Dimsum frozen (72 PCS) - Tanpa Kategori
Masukkan nomor pilihan Anda: 3
Masukkan Quantity: 21
✅ CSRF TOKEN:
XD0oYw5JKRCp1qMG18cKYzJLOJmJWOoM0pjTWWBs
✅ Cookies berhasil dipindahkan
UPDATE URL:
https://larynfood.com/produk-paket
✅ Request update terkirim
========== RESPONSE ==========
STATUS CODE: 200
FINAL URL: https://larynfood.com/produk-paket
✅✅✅ Skenario Tambah Produk Pak

**Edit Produk Paket**

In [ ]:
from selenium.webdriver.support.ui import Select
def test_edit_packet_with_id(driver):
    driver.get("https://larynfood.com/produk-paket")

    names = input("Masukkan Paket yang ingin diedit: ")

    soup = BeautifulSoup(driver.page_source, 'lxml')
    table = soup.find('table', class_='table')



    rows = table.find_all('tr')[1:]

    for row in rows:
        cols = row.find_all('td')
        if not cols: continue

        paket_id = None

        nama = cols[1].get_text(strip=True)

        if nama == names:
            print(f"Match ditemukan: {nama}")
            link_edit = cols[6].find('a', href=re.compile(r'/produk-paket/\d+/edit'))

            paket_id = None
            if link_edit:
                href = link_edit.get('href')

                match = re.search(r'/produk-paket/(\d+)/edit', href)
                if match:
                    paket_id = match.group(1)

            xpath_edit = "//tr[contains(., names)]//a[contains(@href, '/edit')]"
            edit_button = driver.find_element(By.XPATH, xpath_edit)

            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", edit_button)
            time.sleep(0.5)

            edit_button.click()

            print("✅ Berhasil masuk halaman edit stock")

            nama_paket = input("Masukkan Nama Paket: ")
            namaPaket_input = "input[name='nama_paket']"

            kode_paket = input("Masukkan Kode Paket: ")
            kodePaket_input = "input[name='kode_paket']"


            deskripsi = input("Masukkan Deskripsi Paket: ")
            deskripsi_input = "textarea[name='deskripsi']"

            data_paket = {
            nama_paket : namaPaket_input,
            kode_paket : kodePaket_input,
            deskripsi : deskripsi_input
            }


            # Jika ingin dimasukkan ke dalam satu Array (List)
            daftar_paket = [data_paket]

            for data in daftar_paket:
                for nama, input_field in data.items():
                    driver.find_element(By.CSS_SELECTOR, input_field).send_keys(nama)



            dropdown = Select(driver.find_element(By.NAME, "status"))

            # Tampilkan semua pilihan ke terminal
            print("\n--- Daftar Status ---")
            for index, option in enumerate(dropdown.options):
                print(f"{index + 1}. {option.text}")

            # Minta input berdasarkan nomor yang tampil
            pilihan = int(input("\nMasukkan nomor pilihan Anda: "))

            pilih = "valid"

            if 0 < pilihan <= len(dropdown.options):
              dropdown.options[pilihan - 1].click()
              value_terpilih = dropdown.options[pilihan - 1].get_attribute("value")

              # Semua pilihan yang ada di dalam dropdown adalah VALID secara UI
              pilih = value_terpilih
              print(f"✅ Opsi terpilih: {value_terpilih} (Status: {value_terpilih})")
            else:
              print("Pilihan di luar jangkauan!")


            clickTombolItem = "button[id='btnTambahItem']"
            element2 = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, clickTombolItem)))

            # Klik menggunakan JavaScript
            driver.execute_script("arguments[0].click();", element2)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table[id='tableItems']")))
            print("✅ Table Item Berhasil tampil")

            time.sleep(1)

            tombol_tambah = element2

            jml_item = int(input("\nMasukkan Jumlah Item: "))

            items_payload = []
            for i in range(jml_item):
                # 1. Klik tombol tambah (pastikan tombol_tambah didefinisikan dengan benar)
                driver.execute_script("arguments[0].click();", tombol_tambah)
                print(f"➕ Menambah baris ke-{i}...")

                selector_check = f"select[name='items[{i}][stock_gudang_id]']"
                dropdown_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector_check)))

                select_item = Select(dropdown_element)

                print(f"\n--- Daftar Item {i} ---")
                for index, option in enumerate(select_item.options):
                    print(f"{index + 1}. {option.text}")

                pilih2 = int(input("Masukkan nomor pilihan Anda: "))

                if 0 < pilih2 <= len(select_item.options):
                    # FIX 1: Gunakan pilih2, bukan pilihan
                    value_terpilih2 = select_item.options[pilih2 - 1].get_attribute("value")

                    # Pilih di UI agar sinkron
                    select_item.select_by_value(value_terpilih2)

                    # AMBIL DATA-HPP DARI ATRIBUT HTML
                    hpp_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, f"tr:nth-child({i+1}) .hpp-pcs-display")))

                    # Gunakan .text (bukan getText)
                    hpp_text = hpp_element.text or "0"
                else:
                    print("Pilihan di luar jangkauan!")
                    continue

                qty_input = driver.find_element(By.CSS_SELECTOR, f"input[name='items[{i}][qty_per_paket]']")
                qty_input.clear()
                qty = int(input("Masukkan Quantity: "))
                qty_input.send_keys(qty)
                driver.execute_script("arguments[0].dispatchEvent(new Event('input'));", qty_input)

                hpp_satuan = hpp_text.replace("Rp", "").replace(".", "").replace(",", "").strip()


                try:
                    hpp_satuan_int = int(hpp_satuan)
                except ValueError:
                    hpp_satuan_int = 0



                subtotal_item = hpp_satuan_int * qty

                item_baru = {
                    "stock_gudang_id": value_terpilih2,
                    "qty_per_paket": qty,
                    "hpp_satuan": hpp_satuan_int,
                    "subtotal_hpp": subtotal_item
                }

                items_payload.append(item_baru)


            session = requests.Session()


            # =========================
            # AMBIL CSRF TOKEN
            # =========================
            try:

                csrf_token = (
                    driver.find_element(
                        By.NAME,
                        "_token"
                    ).get_attribute("value")
                )

                print("✅ CSRF TOKEN:")
                print(csrf_token)

            except Exception as e:

                print(
                    "❌ Gagal ambil CSRF token"
                )

                print(e)

                return

            # =========================
            # COPY COOKIES SELENIUM
            # =========================
            selenium_cookies = (
                driver.get_cookies()
            )

            for cookie in selenium_cookies:

                session.cookies.set(
                    cookie["name"],
                    cookie["value"]
                )

            print(
                "✅ Cookies berhasil dipindahkan"
            )

            # =========================
            # UPDATE URL
            # =========================
            update_url = (
                f"https://larynfood.com/"
                f"produk-paket/{paket_id}"
            )

            print("UPDATE URL:")
            print(update_url)

            payload = {
              "_token": csrf_token,
              "_method": "PUT",
              "nama_paket": nama_paket,
              "kode_paket": kode_paket,
              "deskripsi": deskripsi,
              "status": pilih, # Pastikan berisi "aktif" atau "nonaktif"
              "items": items_payload,     # Kita gunakan list/array untuk menampung item
          }


            # =========================
            # HEADERS
            # =========================
            headers = {

                "Referer":
                    driver.current_url,

                "User-Agent":
                    driver.execute_script(
                        "return navigator.userAgent;"
                    ),

                "X-Requested-With":
                    "XMLHttpRequest",

                "Accept": "application/json", # WAJIB agar response berbentuk JSON
            "Content-Type": "application/json"
            }

            # =========================
            # SEND REQUEST
            # =========================
            try:

                response = session.post(
                    update_url,
                    json=payload,
                    headers=headers,
                    allow_redirects=True
                )


                print(
                    "✅ Request update terkirim"
                )

            except Exception as e:

                print(
                    "❌ Gagal kirim request"
                )

                print(e)

                return

            # =========================
            # DEBUG RESPONSE
            # =========================
            print("========== RESPONSE ==========")

            print(
                "STATUS CODE:",
                response.status_code
            )

            print(
                "FINAL URL:",
                response.url
            )

            print("==============================")

            # =========================
            # VALIDASI RESULT
            # =========================
            if "/login" in response.url:

                print(
                    "❌ Redirect ke LOGIN"
                )

                print(
                    "⚠️ Session/auth middleware "
                    "masih ditolak"
                )

                with open(
                    "response_login.html",
                    "w",
                    encoding="utf-8"
                ) as f:

                    f.write(response.text)

                print(
                    "✅ HTML response disimpan "
                    "ke response_login.html"
                )

                return

            elif response.status_code == 200:

                print("✅✅✅ Skenario Edit Paket Produk Berhasil dengan waktu " + ' ' + str(time.time()) + ' Detik')

                return

            else:

                print(
                    "⚠️ Response tidak dikenali"
                )

                print(response.text[:1000])

                return


test_edit_packet_with_id(driver)

Masukkan Paket yang ingin diedit: PKT-4005
Match ditemukan: PKT-4005
✅ Berhasil masuk halaman edit stock
Masukkan Nama Paket: Paket Hemat Banget
Masukkan Kode Paket: PKT-4005
Masukkan Deskripsi Paket: -

--- Daftar Status ---
1. Aktif
2. Nonaktif

Masukkan nomor pilihan Anda: 2
✅ Opsi terpilih: nonaktif (Status: nonaktif)
✅ Table Item Berhasil tampil

Masukkan Jumlah Item: 2
➕ Menambah baris ke-0...

--- Daftar Item 0 ---
1. -- Pilih Item Stock Gudang --
2. dimsum (12 PCS) - Dimsum kukus
3. Dimsum frozen (72 PCS) - Tanpa Kategori
Masukkan nomor pilihan Anda: 2
Masukkan Quantity: 23
➕ Menambah baris ke-1...

--- Daftar Item 1 ---
1. -- Pilih Item Stock Gudang --
2. dimsum (12 PCS) - Dimsum kukus
3. Dimsum frozen (72 PCS) - Tanpa Kategori
Masukkan nomor pilihan Anda: 3
Masukkan Quantity: 12
✅ CSRF TOKEN:
XD0oYw5JKRCp1qMG18cKYzJLOJmJWOoM0pjTWWBs
✅ Cookies berhasil dipindahkan
UPDATE URL:
https://larynfood.com/produk-paket/16
✅ Request update terkirim
========== RESPONSE ==========
STATUS 

**Hapus Produk Paket**

In [ ]:
def test_delete_stock_with_id(driver):
    driver.get("https://larynfood.com/produk-paket")

    names = input("Masukkan Paket yang ingin dihapus: ")
    found = False

    while True:
        soup = BeautifulSoup(driver.page_source, 'lxml')
        table = soup.find('table', class_='table')

        if not table: break

        rows = table.find_all('tr')[1:]

        for row in rows:
            cols = row.find_all('td')
            if not cols: continue

            nama = cols[1].get_text(strip=True)

            if nama == names:
                print(f"Match ditemukan: {nama}")

                xpath_delete = f"//tr[contains(., '{names}')]//button[@type='submit']"
                delete_button = driver.find_element(By.XPATH, xpath_delete)

                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", delete_button)
                time.sleep(0.5)

                delete_button.click()

                try:
                    WebDriverWait(driver, 5).until(EC.alert_is_present())
                    alert = driver.switch_to.alert
                    print(f"Isi Alert: {alert.text}")
                    message_input = input('Ok atau Cancel ')
                    if message_input == 'Ok':
                      alert.accept() # Klik OK
                    else:
                      alert.dismiss() # Klik Cancel
                    print(f"🗑️ Paket '{names}' berhasil dihapus.")
                    found = True
                except:
                    print("Tidak ada alert konfirmasi.")

                time.sleep(2)
                print("✅✅✅ Skenario Hapus Paket Berhasil dengan waktu" + ' ' + str(time.time()) + ' Detik')
                return

        # Logika Pagination: Jika tidak ketemu di halaman ini, coba klik 'Next'
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")
            next_button.click()
            time.sleep(2) # Tunggu loading halaman berikutnya
        except:
            print("Nama tidak ditemukan hingga halaman terakhir.")
            break

test_delete_stock_with_id(driver)


Masukkan Paket yang ingin dihapus: PKT-4005
Match ditemukan: PKT-4005
Isi Alert: Yakin ingin menghapus paket ini?
Ok atau Cancel Ok
🗑️ Paket 'PKT-4005' berhasil dihapus.
✅✅✅ Skenario Hapus Paket Berhasil dengan waktu 1778381186.3038394 Detik
